# Area Analysis

Given a center coordinate and a few candidate target species, fetch Xeno-Canto metadata
for the area (50 km radius by default), build a per-species heatmap, and summarize what
else lives there — species/genus/family counts, co-occurring birds (`Recording.also`),
and time-bounded annotations (`Recording.annotation_set`).

**Prerequisites**
- `.env` in this folder with `XC_API_KEY=...`
- `uv sync` in `building/`
- `building/raw_dataset/taxonomy.csv` cached (gets fetched on first `building.taxonomy` import)

In [73]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [74]:
from pathlib import Path

import pyrootutils
from xenocanto3 import Quality

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

# Parameters

In [75]:
LAT, LON = 48.8566, 2.3522            # Paris
RADIUS_KM = 50
SPECIES_NAMES = ["Emberiza calandra"]
MIN_QUALITY = Quality.B
MIN_RECORDINGS = 10                   # drop species with fewer than this many recordings in the area from the taxonomy breakdown

# BirdNET pass
MAX_BIRDNET_RECORDINGS = 500          # cap downloads — set to None to process all area recordings
BIRDNET_MIN_CONFIDENCE = 0.85         # detection threshold (download.py default is 0.92)

# Bounding box

In [76]:
from building.analysis import bbox_from_radius

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
print(f"bbox (lat_min, lon_min, lat_max, lon_max): {tuple(round(x, 4) for x in bbox)}")
print(f"lat span: {bbox[2] - bbox[0]:.3f}°   lon span: {bbox[3] - bbox[1]:.3f}°")

bbox (lat_min, lon_min, lat_max, lon_max): (48.4061, 1.6676, 49.3071, 3.0368)
lat span: 0.901°   lon span: 1.369°


# Per-species global heatmap

One toggleable layer per input species; the analysis area is overlaid as a black rectangle.

In [77]:
from building.analysis import fetch_species_global, recordings_to_df, build_heatmap, ANALYSIS_DIR

global_recs = await fetch_species_global(SPECIES_NAMES, min_quality=MIN_QUALITY)
per_species_df = {name: recordings_to_df(recs) for name, recs in global_recs.items()}

for name, df in per_species_df.items():
    print(f"{name:30s} {len(df):5d} recordings (with coords: {df['lat'].notna().sum()})")

heatmap = build_heatmap(per_species_df, center=(LAT, LON), bbox=bbox)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
heatmap.save(str(ANALYSIS_DIR / "heatmap.html"))
heatmap

Emberiza calandra                229 recordings (with coords: 228)


# Recordings inside the area

In [78]:
from building.analysis import fetch_area_recordings

area_recs = await fetch_area_recordings(bbox, min_quality=MIN_QUALITY)
area_df = recordings_to_df(area_recs)
print(f"{len(area_df)} recordings, {area_df['scientific_name'].nunique()} unique species")

764 recordings, 145 unique species


# Are the target species present in the area?

In [79]:
from building.analysis import presence_in_area

presence_in_area(SPECIES_NAMES, area_df)

,scientific_name,recordings_in_area,qualities,mean_length_sec
0,Emberiza calandra,2,A,59.0


# Taxonomy breakdown of the area

How many orders / families / genera / species does the area hold? Useful for scoping a future training collection.

In [80]:
from building.analysis import taxonomy_breakdown

tax = taxonomy_breakdown(area_df, min_recordings=MIN_RECORDINGS)
totals = tax["totals"].iloc[0]
print(
    f"{totals['n_recordings']} recordings  |  "
    f"{totals['n_species']} species across {totals['n_genera']} genera, "
    f"{totals['n_families']} families, {totals['n_orders']} orders"
    f"  (species with ≥ {MIN_RECORDINGS} recordings)"
)
if totals["n_unmatched_in_taxonomy"]:
    print(f"({totals['n_unmatched_in_taxonomy']} recordings could not be matched to the eBird taxonomy)")
tax["totals"]

403 recordings  |  25 species across 20 genera, 17 families, 5 orders  (species with ≥ 10 recordings)
(41 recordings could not be matched to the eBird taxonomy)


,n_orders,n_families,n_genera,n_species,n_recordings,n_unmatched_in_taxonomy
0,5,17,20,25,403,41


In [81]:
tax["by_order"]

,order,species,recordings
0,Passeriformes,17,275
1,NaN,3,41
2,Strigiformes,2,30
3,Piciformes,1,22
4,Psittaciformes,1,21
5,Coraciiformes,1,14


In [82]:
tax["by_family"].head(20)

,order,family,species,recordings
0,Passeriformes,Sylviidae (Sylviid Warblers and Allies),3,48
1,NaN,NaN,3,41
2,Passeriformes,Acrocephalidae (Reed Warblers and Allies),2,39
3,Passeriformes,Muscicapidae (Old World Flycatchers),2,30
4,Passeriformes,Locustellidae (Grassbirds and Allies),2,28
5,Passeriformes,Motacillidae (Wagtails and Pipits),1,26
6,Piciformes,Picidae (Woodpeckers),1,22
7,Passeriformes,"Paridae (Tits, Chickadees, and Titmice)",1,22
8,Psittaciformes,Psittaculidae (Old World Parrots),1,21
9,Passeriformes,Regulidae (Kinglets),1,20


In [83]:
tax["by_genus"].head(20)

,family,genus,species,recordings
0,NaN,NaN,3,41
1,Sylviidae (Sylviid Warblers and Allies),Curruca,2,34
2,Locustellidae (Grassbirds and Allies),Locustella,2,28
3,Acrocephalidae (Reed Warblers and Allies),Hippolais,1,26
4,Motacillidae (Wagtails and Pipits),Anthus,1,26
5,Picidae (Woodpeckers),Dendrocoptes,1,22
6,"Paridae (Tits, Chickadees, and Titmice)",Parus,1,22
7,Psittaculidae (Old World Parrots),Psittacula,1,21
8,Regulidae (Kinglets),Regulus,1,20
9,Certhiidae (Treecreepers),Certhia,1,18


# Co-occurring species

From each recording's `also` field — species heard in the background. Useful for discovering candidates that don't have their own recordings in the area but show up alongside others.

In [84]:
from building.analysis import co_occurrence_counts

co_occurrence_counts(area_df, top_n=30)

,species,count
0,Fringilla coelebs,11
1,Phylloscopus collybita,10
2,Corvus corone,8
3,Troglodytes troglodytes,6
4,Turdus merula,6
5,Psittacula krameri,5
6,Sylvia atricapilla,4
7,Poecile palustris,4
8,Parus major,4
9,Erithacus rubecula,3


# Annotations

Time-bounded annotations attached to recordings (rare — most XC recordings are not annotated, but the ones that are can be useful training labels).

In [85]:
from building.analysis import annotation_summary

ann = annotation_summary(area_df)
print(f"{len(ann['annotations'])} annotations across {len(ann['by_species'])} species")
ann["by_species"].head(20)

0 annotations across 0 species


,annotated_species,count


In [86]:
ann["annotations"].head(20)

""


# BirdNET pass on area recordings

Download the area's recordings and run BirdNET on each. Some birds are present in the audio but were not flagged by the recordist in XC's `also` field — BirdNET surfaces them.

Files cache to `raw_dataset/analysis/audio/` and downloads are skipped on re-run. Delete that folder to reclaim disk.

In [87]:
from building.analysis import birdnet_area_analysis

bn = await birdnet_area_analysis(
    area_df,
    max_recordings=MAX_BIRDNET_RECORDINGS,
    min_confidence=BIRDNET_MIN_CONFIDENCE,
)
det = bn["detections"]
print(
    f"{len(det)} detections across {det['xc_id'].nunique()} recordings  |  "
    f"{bn['by_species'].shape[0]} unique species detected  |  "
    f"{bn['hidden'].shape[0]} species not declared in XC `also`"
)

BirdNET analyse:   0%|          | 0/500 [00:00<?, ?it/s]

read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording XC528945.mp3
read_audio_data
read_audio_data: complete, read  5 chunks.
analyze_recording XC484893.mp3
read_audio_data
read_audio_data: complete, read  11 chunks.
analyze_recording XC484892.mp3
read_audio_data
read_audio_data: complete, read  4 chunks.
analyze_recording XC484891.mp3
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording XC386855.mp3
read_audio_data
read_audio_data: complete, read  5 chunks.
analyze_recording XC386854.mp3
read_audio_data
read_audio_data: complete, read  44 chunks.
analyze_recording XC360691.mp3
read_audio_data
read_audio_data: complete, read  3 chunks.
analyze_recording XC360690.mp3
read_audio_data
read_audio_data: complete, read  38 chunks.
analyze_recording XC464102.mp3
read_audio_data
read_audio_data: complete, read  7 chunks.
analyze_recording XC560729.mp3
read_audio_data
read_audio_data: complete, read  12 chunks.
analyze_recording XC560723.mp3
read_a

### All detected species

In [88]:
bn["by_species"].head(40)

,detected_species,n_detections,n_recordings,n_undeclared,mean_conf,max_conf
67,Locustella naevia,598,14,1,0.974,1.000
15,Athene noctua,515,12,0,0.986,1.000
103,Sylvia borin,469,11,7,0.972,1.000
66,Locustella luscinioides,444,20,13,0.988,1.000
59,Hippolais polyglotta,284,27,7,0.965,1.000
76,Parus major,244,21,10,0.966,1.000
95,Regulus ignicapilla,226,21,33,0.977,1.000
1,Acrocephalus palustris,214,17,9,0.932,0.999
35,Curruca communis,203,21,2,0.961,1.000
45,Dendrocoptes medius,188,23,5,0.980,1.000


### Hidden species — detected by BirdNET but not declared in XC `also`

`n_undeclared` = how many BirdNET detections of this species were in recordings where the recordist did **not** list it (neither as primary nor in `also`). High values = systematically under-reported.

In [89]:
bn["hidden"].head(40)

,detected_species,n_detections,n_recordings,n_undeclared,mean_conf,max_conf
0,Regulus ignicapilla,226,21,33,0.977,1.000
1,Cyanistes caeruleus,77,8,21,0.946,0.999
2,Locustella luscinioides,444,20,13,0.988,1.000
3,Caprimulgus europaeus,41,6,13,0.986,1.000
4,Dendrocopos major,157,9,13,0.943,0.997
5,Phylloscopus collybita,42,5,13,0.976,0.999
6,Corvus cornix,11,8,11,0.891,0.948
7,Sylvia atricapilla,60,15,11,0.936,0.996
8,Poecile palustris,77,9,10,0.974,1.000
9,Parus major,244,21,10,0.966,1.000


# All area species — split by source

For each species in the area, three mutually exclusive per-recording counts:

- **`xc_primary`** — recordings whose declared primary species is this one
- **`xc_also`** — recordings that list this species in `also`
- **`birdnet_only`** — recordings where BirdNET detected this species and the recordist did **not** list it (neither primary nor in `also`)

Filter: `xc_primary + xc_also + birdnet_only ≥ MIN_RECORDINGS`. Sorted by total desc.

> The `birdnet_only` column is sparse unless `MAX_BIRDNET_RECORDINGS` covers most of the area.

In [90]:
from building.analysis import combined_species_table

combined = combined_species_table(area_df, bn["detections"], min_recordings=MIN_RECORDINGS)
print(f"{len(combined)} species (≥ {MIN_RECORDINGS} recordings of any source)")
combined

38 species (≥ 10 recordings of any source)


,scientific_name,xc_primary,xc_also,birdnet_only,total
0,Hippolais polyglotta,26,0,5,31
1,Psittacula krameri,21,5,3,29
2,Parus major,22,4,3,29
3,Anthus spinoletta,26,0,2,28
4,Dendrocoptes medius,22,0,3,25
5,Curruca communis,22,0,2,24
6,Fringilla coelebs,10,11,2,23
7,Sylvia atricapilla,14,4,4,22
8,Regulus ignicapilla,20,0,2,22
9,Certhia brachydactyla,18,1,3,22
